# Size Distribution in PyPath

## Overview
The size distribution system models population size structure using Von Bertalanffy growth functions. This notebook demonstrates size classes, population size distributions (PSD), and VBGF examples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypath.core.params import SizeClass, SizeDistribution
from pypath.core.params import vb_length, vb_age

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Creating SizeClass Objects

SizeClass represents a single size class with boundaries:

In [ ]:
# Create size classes for Cod
juvenile = SizeClass(
    name='Juvenile',
    min_length_cm=5,
    max_length_cm=30,
    proportion=0.2
)

adult = SizeClass(
    name='Adult',
    min_length_cm=30,
    max_length_cm=100,
    proportion=0.6
)

old = SizeClass(
    name='Old',
    min_length_cm=100,
    max_length_cm=200,
    proportion=0.2
)

print('Size Classes:')
print(f'  {juvenile.name}: {juvenile.min_length_cm}-{juvenile.max_length_cm} cm')
print(f'  {adult.name}: {adult.min_length_cm}-{adult.max_length_cm} cm')
print(f'  {old.name}: {old.min_length_cm}-{old.max_length_cm} cm')

## Part 2: Von Bertalanffy Growth Function (VBGF)

Model fish growth using VBGF parameters:

In [ ]:
# VBGF parameters for Atlantic Cod
Linf = 150.0  # Asymptotic length (cm)
K = 0.15      # Growth rate (year^-1)
t0 = 0.0      # Theoretical age at zero length (year)

# Calculate length at age
ages = np.linspace(0, 20, 101)
lengths = vb_length(ages, Linf=Linf, K=K, t0=t0)

# Plot growth curve
plt.figure(figsize=(10, 6))
plt.plot(ages, lengths, linewidth=2, label='Cod VBGF')
plt.xlabel('Age (years)')
plt.ylabel('Length (cm)')
plt.title(f'Von Bertalanffy Growth: Linf={Linf}, K={K}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

## Part 3: Age-Length Conversion

Calculate age from length using inverse VBGF:

In [ ]:
# Calculate age at specific lengths
test_lengths = np.array([30, 50, 75, 100, 130])
ages_at_length = vb_age(test_lengths, Linf=Linf, K=K, t0=t0)

print('Age-Length Table:')
for length, age in zip(test_lengths, ages_at_length):
    print(f'  {length:.0f} cm: {age:.2f} years')

## Part 4: Creating Size Distribution Container

Manage population size structure:

In [ ]:
# Create size distribution for Cod
cod_size_dist = SizeDistribution(
    group_name='Cod',
    linf=Linf,
    k=K,
    t0=t0
)

# Add size classes
cod_size_dist.add_class(juvenile)
cod_size_dist.add_class(adult)
cod_size_dist.add_class(old)

print(f'Size distribution created for {cod_size_dist.group_name}')
print(f'Number of classes: {len(cod_size_dist)}')
print(f'Total proportion: {cod_size_dist.total_proportion():.1%}')

## Part 5: Population Size Distribution (PSD)

Generate PSD from parameters:

In [ ]:
# Generate PSD using current distribution
psd = cod_size_dist.get_psd(total_biomass=1000.0)
print('Population Size Distribution:')
print(psd)

# Visualize PSD
psd_df = pd.DataFrame([
    {'Class': 'Juvenile', 'Biomass': psd[0]},
    {'Class': 'Adult', 'Biomass': psd[1]},
    {'Class': 'Old', 'Biomass': psd[2]}
])

plt.figure(figsize=(8, 5))
plt.bar(psd_df['Class'], psd_df['Biomass'], color='steelblue')
plt.ylabel('Biomass (t)')
plt.title('Population Size Distribution - Cod')
plt.tight_layout()

## Part 6: Validation

In [ ]:
# Validate size distribution
is_valid = cod_size_dist.validate()
print(f'Size distribution is valid: {is_valid}')

# Check for overlapping size classes
for i, sc in enumerate(cod_size_dist.classes()):
    print(f'Class {i}: {sc.min_length_cm}-{sc.max_length_cm} cm ({sc.proportion:.0%})')

## Part 7: DataFrame Serialization

In [ ]:
# Convert to DataFrame
size_df = cod_size_dist.to_dataframe()
print('Size Distribution DataFrame:')
print(size_df)

# Save to CSV
size_df.to_csv('size_distribution.csv', index=False)
print('\nSaved to size_distribution.csv')

## Performance Tips

- VBGF calculations are vectorized using NumPy
- Size class lookup is O(1) by name
- PSD generation is O(n) where n = number of size classes
- Pre-compute VBGF curves and cache for multiple populations